# Практика: RFM плюс производные признаки


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


**Центральная идея:** Производный признак полезен только при ясном правиле, корректной агрегации и документированном масштабе.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_05_shop_feature_engineering/data/" + name

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## 1. Базовый RFM

Соберите канонический RFM через одну цепочку.

**Зачем:** Производный признак полезен только при ясном правиле, корректной агрегации и документированном масштабе. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
merged=orders.merge(payments,on="order_id",validate="one_to_one")
rfm=None  # TODO
assert len(rfm)==778 and {"Recency","Frequency","Monetary"}<=set(rfm)


## 2. Бинарная оплата

Создайте is_card перед агрегацией.

**Зачем:** Производный признак полезен только при ясном правиле, корректной агрегации и документированном масштабе. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
merged["is_card"]=None  # TODO
assert set(merged["is_card"])=={0,1}


## 3. Доля card

Агрегируйте среднее is_card на клиента и присоедините.

**Зачем:** Производный признак полезен только при ясном правиле, корректной агрегации и документированном масштабе. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
card_share=None; rfm_plus=None  # TODO
assert len(rfm_plus)==len(rfm) and rfm_plus["share_card"].between(0,1).all()


## 4. Средний срок доставки

Создайте и агрегируйте days_to_deliver.

**Зачем:** Производный признак полезен только при ясном правиле, корректной агрегации и документированном масштабе. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
merged["days_to_deliver"]=None; delivery_mean=None  # TODO
rfm_plus=rfm_plus.merge(delivery_mean.rename("avg_days_to_deliver"),on="customer_id",how="left")
assert rfm_plus["avg_days_to_deliver"].notna().sum()>700


## 5. Штат клиента

Присоедините категорию customer_state после агрегации.

**Зачем:** Производный признак полезен только при ясном правиле, корректной агрегации и документированном масштабе. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
rfm_plus=None  # TODO: merge customers
assert len(rfm_plus)==778 and rfm_plus["customer_state"].notna().all()


## 6. Корреляция F и M

Рассчитайте Pearson и напишите осторожную интерпретацию.

**Зачем:** Производный признак полезен только при ясном правиле, корректной агрегации и документированном масштабе. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
corr_fm=None; CORR_NOTE=""  # TODO
assert -1<=corr_fm<=1 and len(CORR_NOTE)>=190
assert "причин" in CORR_NOTE.lower()


## 7. Масштабированный score

Нормируйте F и M средними; меньший Recency должен повышать score.

**Зачем:** Производный признак полезен только при ясном правиле, корректной агрегации и документированном масштабе. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
scored=rfm_plus.copy()
scored["score"]=None  # TODO
assert np.isfinite(scored["score"]).all()
assert scored.nlargest(1,"score").index.size==1


## 8. Контракт RFM+

Проверьте границы, уникальность и отсутствие churn.

**Зачем:** Производный признак полезен только при ясном правиле, корректной агрегации и документированном масштабе. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
checks={"unique":None,"frequency":None,"money":None,"share":None,"no_churn":None}  # TODO
assert set(checks.values())=={True}
